In [1]:
"""
2026 FIFA 월드컵 예측용 입력 데이터 생성 (v3 - 최종)
=======================================================
[설계 원칙]
- 모든 팀명은 우리 results_preprocessed.csv 기준 (카글 팀명 따르지 않음)
- 카글(test.csv)은 wc_participations, wc_titles 두 가지만 참고
  → 역대 누적 기록이라 우리 데이터로 직접 계산 불가한 항목

[카글 검증 결과 및 이슈 정리]
① wc_participations: 컬럼명이 'world_cup_participations_before'로
   2026 대회 미포함 기준 → 우리 모델도 동일 방식이므로 그대로 사용
② wc_titles: 브라질5/독일4/이탈리아4/아르헨티나3/프랑스2/우루과이2/잉글랜드1/스페인1
   → 검증 완료, 정확
③ is_host: 카글 사용 안 함 → 미국/캐나다/멕시코만 1, 하드코딩으로 처리
④ continent: 카글이 Australia를 Oceania로 잘못 분류
   → 카글 사용 안 함, results_preprocessed에서 직접 추출 (우리 데이터는 AFC로 정확히 인코딩됨)

[4년 성적 기간 이슈]
- 카글은 카타르 WC 종료(2022-12-20) 이후를 4년 성적으로 정의 → 우리와 다름
- 우리는 WC 개막 직전 정확히 4년(2022-06-10 ~ 2026-06-10)으로 일관되게 계산
- 학습 데이터와 동일한 방식으로 계산해야 하므로 우리 기준 유지

[팀명 이슈]
- South Korea → Korea Republic (우리 데이터 표기)
- Czech Republic → Czechia
- Ivory Coast → Cote d'Ivoire
- 카글 'Cura?o' → Curacao (인코딩 깨짐 수정)

출력: wc2026_matches.csv — 72경기 × (메타4 + 피처52)
"""

import pandas as pd
import numpy as np
import json

# ============================================================
# 0. 경로 설정
# ============================================================
BASE        = "/content/drive/MyDrive/[미래융합교육원] 1팀 공유사항/1차 머신러닝 프로젝트/@프로젝트 자료/03. 프로젝트 자료/data"
SAVE_PATH   = f"{BASE}/preprocessed"
KAGGLE_PATH = f"{BASE}/FIFA World Cup Dataset/test.csv"

with open(f"{SAVE_PATH}/features.json") as f:
    FEATURES = json.load(f)

results = pd.read_csv(f"{SAVE_PATH}/results_preprocessed.csv", parse_dates=["date"])
print(f"✅ 로드 완료: {len(results):,}경기 ({results['date'].min().date()} ~ {results['date'].max().date()})")

# ============================================================
# 1. 카글 로드 — wc_participations, wc_titles 전용
#    [주의] 카글 팀명을 우리 팀명으로 변환 후 인덱싱
# ============================================================
kaggle_raw = pd.read_csv(KAGGLE_PATH)
kaggle_raw["team"] = kaggle_raw["team"].str.strip().replace({
    "Cura?o":        "Curacao",        # 인코딩 깨짐 수정
    "South Korea":   "Korea Republic", # 우리 표기로 통일
    "Czech Republic":"Czechia",        # 우리 표기로 통일
    "Ivory Coast":   "Cote d'Ivoire",  # 우리 표기로 통일
})
kaggle = kaggle_raw.set_index("team")  # 우리 팀명 기준 인덱싱

# ============================================================
# 2. 2026 WC 조편성 — 우리 팀명 기준
#    [참고] 2026 WC 공식 조편성 (2025-12-05 워싱턴 D.C. 추첨)
# ============================================================
GROUPS = {
    "A": ["Korea Republic",  "Czechia",                "Mexico",        "South Africa"],
    "B": ["Canada",          "Switzerland",            "Qatar",         "Bosnia and Herzegovina"],
    "C": ["Brazil",          "Morocco",                "Scotland",      "Haiti"],
    "D": ["United States",   "Paraguay",               "Australia",     "Turkey"],
    "E": ["Germany",         "Curacao",                "Cote d'Ivoire", "Ecuador"],
    "F": ["Netherlands",     "Japan",                  "Tunisia",       "Sweden"],
    "G": ["Belgium",         "Egypt",                  "Iran",          "New Zealand"],
    "H": ["Spain",           "Cape Verde",             "Saudi Arabia",  "Uruguay"],
    "I": ["France",          "Senegal",                "Norway",        "Iraq"],
    "J": ["Argentina",       "Algeria",                "Austria",       "Jordan"],
    "K": ["Portugal",        "Colombia",               "Uzbekistan",    "DR Congo"],
    "L": ["England",         "Croatia",                "Ghana",         "Panama"],
}

# 개최국 — 하드코딩 (카글 미사용, 3개국만 명확하므로)
HOST_TEAMS = {"United States", "Canada", "Mexico"}

def make_group_pairs(teams):
    """각 조 라운드로빈 6경기 생성 (FIFA 공식 대진 순서)"""
    t = teams
    return [
        (t[0], t[1], 1), (t[2], t[3], 1),  # 라운드 1
        (t[0], t[2], 2), (t[1], t[3], 2),  # 라운드 2
        (t[0], t[3], 3), (t[2], t[1], 3),  # 라운드 3 (동시 킥오프)
    ]

# ============================================================
# 3. 4년 성적 직접 계산 (우리 데이터 기준)
#    기간: 2022-06-10 ~ 2026-06-10 (WC 개막 직전 정확히 4년)
#    [이슈] 카글은 카타르WC 종료(2022-12-20) 이후 기준으로 계산해
#           우리 값보다 경기 수가 적게 나옴 → 우리 기준으로 일관성 유지
# ============================================================
DATE_END   = pd.Timestamp("2026-06-10")
DATE_START = DATE_END - pd.DateOffset(years=4)  # 2022-06-10
print(f"📅 4년 성적 기간: {DATE_START.date()} ~ {DATE_END.date()}")

def calc_4y_stats(team: str) -> dict:
    mask = (
        ((results["home_team"] == team) | (results["away_team"] == team)) &
        (results["date"] >= DATE_START) &
        (results["date"] <= DATE_END)
    )
    sub = results[mask]

    wins = losses = draws = 0
    gs = gr = 0.0
    for _, r in sub.iterrows():
        if r["home_team"] == team:
            g1, g2 = r["home_score"], r["away_score"]
        else:
            g1, g2 = r["away_score"], r["home_score"]
        gs += g1; gr += g2
        if g1 > g2:   wins   += 1
        elif g1 < g2: losses += 1
        else:         draws  += 1

    m = wins + losses + draws
    return {
        "goals_scored_last_4y":   gs,
        "goals_received_last_4y": gr,
        "wins_last_4y":           wins,
        "losses_last_4y":         losses,
        "draws_last_4y":          draws,
        "matches_last_4y":        m,
        "goal_diff_last_4y":      gs - gr,
        "win_rate_4y":            wins / m if m > 0 else 0.0,
        "goals_per_match_4y":     gs / m if m > 0 else 0.0,
    }

# 48팀 사전 계산
all_teams     = [t for teams in GROUPS.values() for t in teams]
our_teams_set = set(results["home_team"]) | set(results["away_team"])

print("\n⏳ 48팀 4년 성적 계산 중...")
team_stats = {}
for team in all_teams:
    if team not in our_teams_set:
        # [주의] 팀명 매핑 오류 또는 신규 팀(데이터 없음)일 경우
        print(f"  ⚠️  '{team}' 우리 데이터에 없음 → 0 대체 (팀명 확인 필요)")
        team_stats[team] = {k: 0 for k in [
            "goals_scored_last_4y","goals_received_last_4y","wins_last_4y",
            "losses_last_4y","draws_last_4y","matches_last_4y",
            "goal_diff_last_4y","win_rate_4y","goals_per_match_4y"
        ]}
    else:
        team_stats[team] = calc_4y_stats(team)
print("✅ 완료")

# ============================================================
# 4. Elo 추출 — 우리 데이터에서 대회 직전 최신값
# ============================================================
def get_latest_elo(team: str) -> float:
    sub  = results[results["date"] <= DATE_END]
    home = sub[sub["home_team"] == team].sort_values("date")
    away = sub[sub["away_team"] == team].sort_values("date")

    elo, latest = np.nan, pd.Timestamp("1900-01-01")
    if len(home) > 0 and home.iloc[-1]["date"] > latest:
        elo, latest = home.iloc[-1]["home_elo"], home.iloc[-1]["date"]
    if len(away) > 0 and away.iloc[-1]["date"] > latest:
        elo, latest = away.iloc[-1]["away_elo"], away.iloc[-1]["date"]
    return round(float(elo), 1) if not np.isnan(elo) else np.nan

# ============================================================
# 5. 폼 추출 — 우리 데이터에서 대회 직전 최신값
#    결측 시: 1.0(무승부 승점) 대체 + missing 플래그 1
#    (학습 데이터 전처리 방식과 동일)
# ============================================================
def get_latest_form(team: str) -> tuple:
    sub  = results[results["date"] <= DATE_END]
    home = sub[sub["home_team"] == team].sort_values("date")
    away = sub[sub["away_team"] == team].sort_values("date")

    form, latest = np.nan, pd.Timestamp("1900-01-01")
    if len(home) > 0 and "home_form" in home.columns:
        val = home.iloc[-1].get("home_form", np.nan)
        if not pd.isna(val) and home.iloc[-1]["date"] > latest:
            form, latest = val, home.iloc[-1]["date"]
    if len(away) > 0 and "away_form" in away.columns:
        val = away.iloc[-1].get("away_form", np.nan)
        if not pd.isna(val) and away.iloc[-1]["date"] > latest:
            form, latest = val, away.iloc[-1]["date"]

    return (round(float(form), 4), 0) if not pd.isna(form) else (1.0, 1)

# ============================================================
# 6. 대륙 원핫 인코딩 추출 — 우리 데이터에서 직접
#    [이슈] 카글은 Australia를 Oceania로 잘못 분류
#           우리 results_preprocessed는 AFC로 정확히 인코딩됨
#           → 카글 미사용, 우리 데이터에서 직접 뽑아옴
# ============================================================
CONFS         = ["AFC", "CAF", "CONCACAF", "CONMEBOL", "OFC", "UEFA"]
CONTINENT_COLS = {
    side: [f"{side}_continent_{c}" for c in CONFS]
    for side in ["home", "away"]
}

def get_continent_encoding(team: str) -> dict:
    """
    results_preprocessed에서 해당 팀의 대륙 원핫 인코딩 추출.
    홈/원정 무관하게 가장 최근 경기에서 가져옴.
    """
    sub  = results[results["date"] <= DATE_END]
    home = sub[sub["home_team"] == team].sort_values("date")
    away = sub[sub["away_team"] == team].sort_values("date")

    # 홈 경기에서 추출
    if len(home) > 0:
        row = home.iloc[-1]
        cols_exist = [c for c in CONTINENT_COLS["home"] if c in row.index]
        if cols_exist:
            return {
                c.replace("home_continent_", ""): bool(row[c])
                for c in cols_exist
            }

    # 원정 경기에서 추출 (홈 경기 없을 경우)
    if len(away) > 0:
        row = away.iloc[-1]
        cols_exist = [c for c in CONTINENT_COLS["away"] if c in row.index]
        if cols_exist:
            return {
                c.replace("away_continent_", ""): bool(row[c])
                for c in cols_exist
            }

    # 데이터 없을 경우 — 전부 False
    print(f"  ⚠️  '{team}' 대륙 인코딩 없음 → 전부 0 대체")
    return {c: False for c in CONFS}

# ============================================================
# 7. 단일 팀 피처 딕셔너리 생성
# ============================================================
def build_team_feats(team: str, side: str) -> dict:
    stats      = team_stats[team]
    elo        = get_latest_elo(team)
    form, miss = get_latest_form(team)
    continent  = get_continent_encoding(team)

    feats = {
        f"{side}_elo":          elo,
        f"{side}_form":         form,
        f"{side}_form_missing": miss,
        f"{side}_is_host":      1 if team in HOST_TEAMS else 0,
    }

    # 4년 성적 (우리 계산값)
    for col, val in stats.items():
        feats[f"{side}_{col}"] = val

    # 대륙 원핫 (우리 데이터에서 추출)
    for c in CONFS:
        feats[f"{side}_continent_{c}"] = 1 if continent.get(c, False) else 0

    # wc_participations, wc_titles (카글 참고 — 역대 누적값)
    # [검증 완료] wc_participations: 2026 미포함 기준으로 우리 방식과 일치
    # [검증 완료] wc_titles: 실제 우승 횟수와 일치
    if team in kaggle.index:
        krow = kaggle.loc[team]
        if isinstance(krow, pd.DataFrame):
            krow = krow.iloc[0]  # 중복 행 방지
        feats[f"{side}_wc_participations"] = int(krow["world_cup_participations_before"])
        feats[f"{side}_wc_titles"]         = int(krow["world_cup_titles_before"])
    else:
        # [주의] 카글에 없는 팀 → 0 대체 (팀명 매핑 확인 필요)
        print(f"  ⚠️  카글에 '{team}' 없음 → wc_participations/titles 0 대체")
        feats[f"{side}_wc_participations"] = 0
        feats[f"{side}_wc_titles"]         = 0

    return feats

# ============================================================
# 8. 72경기 피처 조립
# ============================================================
print("\n⏳ 72경기 피처 조립 중...")
rows = []
for group, teams in GROUPS.items():
    for t1, t2, rd in make_group_pairs(teams):
        row = {
            "group":     group,
            "round":     rd,
            "home_team": t1,
            "away_team": t2,
            "neutral":   1,  # 조별리그 전경기 중립 처리
            "is_wc":     1,
        }
        row.update(build_team_feats(t1, "home"))
        row.update(build_team_feats(t2, "away"))

        # Gap 피처 (home - away)
        row["delta_elo"]              = row["home_elo"]                - row["away_elo"]
        row["gap_form"]               = row["home_form"]               - row["away_form"]
        row["gap_goal_diff"]          = row["home_goal_diff_last_4y"]  - row["away_goal_diff_last_4y"]
        row["gap_wins"]               = row["home_wins_last_4y"]       - row["away_wins_last_4y"]
        row["gap_win_rate_4y"]        = row["home_win_rate_4y"]        - row["away_win_rate_4y"]
        row["gap_goals_per_match_4y"] = row["home_goals_per_match_4y"] - row["away_goals_per_match_4y"]
        row["gap_wc_participations"]  = row["home_wc_participations"]  - row["away_wc_participations"]
        row["gap_wc_titles"]          = row["home_wc_titles"]          - row["away_wc_titles"]

        rows.append(row)

wc2026 = pd.DataFrame(rows)
print("✅ 완료")

# ============================================================
# 9. FEATURES 순서 정렬 + 누락 피처 확인
# ============================================================
meta_cols     = ["group", "round", "home_team", "away_team"]
missing_feats = [f for f in FEATURES if f not in wc2026.columns]

if missing_feats:
    print(f"\n⚠️  누락 피처 {len(missing_feats)}개 — 확인 필요:")
    for f in missing_feats:
        print(f"   - {f}")
        wc2026[f] = np.nan
else:
    print("✅ 52개 피처 전부 생성 완료")

X_wc2026 = wc2026[meta_cols + FEATURES]

# ============================================================
# 10. 저장
# ============================================================
OUTPUT = f"{SAVE_PATH}/wc2026_matches.csv"
X_wc2026.to_csv(OUTPUT, index=False, encoding="utf-8-sig")
print(f"\n✅ 저장 완료: {OUTPUT}")
print(f"   {len(X_wc2026)}경기 × {X_wc2026.shape[1]}컬럼")

# ============================================================
# 11. 검증 출력
# ============================================================
preview = ["home_team","away_team","delta_elo","home_elo","away_elo",
           "home_form","away_form","gap_win_rate_4y",
           "home_wc_participations","away_wc_participations",
           "home_wc_titles","away_wc_titles",
           "home_continent_AFC","home_continent_UEFA"]

print("\n📋 Group A 미리보기:")
print(X_wc2026[X_wc2026["group"]=="A"][preview].to_string(index=False))

print("\n📋 한국 전 경기:")
kor = (X_wc2026["home_team"]=="Korea Republic") | (X_wc2026["away_team"]=="Korea Republic")
print(X_wc2026[kor][preview].to_string(index=False))

print("\n📋 Australia 대륙 인코딩 확인 (AFC여야 정상):")
aus = (X_wc2026["home_team"]=="Australia") | (X_wc2026["away_team"]=="Australia")
cont_cols = ["home_team","away_team",
             "home_continent_AFC","home_continent_OFC",
             "away_continent_AFC","away_continent_OFC"]
print(X_wc2026[aus][cont_cols].to_string(index=False))

print("\n📋 NaN 확인:")
nan_cols = X_wc2026[FEATURES].isna().sum()
nan_cols = nan_cols[nan_cols > 0]
print(nan_cols if len(nan_cols) > 0 else "NaN 없음 ✅")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/[미래융합교육원] 1팀 공유사항/1차 머신러닝 프로젝트/@프로젝트 자료/03. 프로젝트 자료/data/preprocessed/features.json'